# Suite TRN — Translators

Every pair of dialects has its own bidirectional translator (`mbse.Expressions.Translators`), declared as rules
that pair two patterns with holes. Translating co-traverses each rule's pattern with the expression to bind the
holes, then instantiates the other side, so every node is translated once and sharing is kept. A dialect with no
binding gets a let's value substituted for its name. Translations keep an expression's form; its value follows
each dialect's rules, and this suite shows where those agree and where they do not.

In [ ]:
import itertools

import numpy as np

from mbse.Expressions import Evaluators as V, Expressions as E, Translators
from mbse.Expressions.Dialects.Excel import Evaluators as XV, Expressions as X
from mbse.Expressions.Dialects.Matlab import Evaluators as MV, Expressions as M
from mbse.Expressions.Dialects.Numpy import Evaluators as NV, Expressions as N
from mbse.Expressions.Framework import Expressions as F
from mbse.Expressions.Framework.Translators import Hole, Inline, Pairwise, Pattern as P, Rule, holes, renames
from mbse.Schemas.Framework import Proxies, Schemas as S
from support import raises, text

BASIC, NUMPY, MATLAB, EXCEL = E.DIALECT, N.DIALECT, M.DIALECT, X.DIALECT
DIALECTS = [BASIC, NUMPY, MATLAB, EXCEL]
EVALUATE = {BASIC: V.OfAny, NUMPY: NV.OfAny, MATLAB: MV.OfAny, EXCEL: XV.OfAny}
RENDER = {NUMPY: N.render, MATLAB: M.render, EXCEL: X.render}
this = E.variable("this")
age = this.age
core = age.ge(18).and_(age.lt(65).or_(this.has("email").not_())).and_(this.name.ne("x")).or_(
    age.add(1).sub(2).mul(3).neg().gt(0).and_(age.eq(1).not_())).and_(this.has("name").eq(True)).and_(
    E.literal(1.5).le(2.5))  # every translatable operation and literal type, operands of one type, no let
rule = E.let_("a", age, E.variable("a").ge(18).implies(this.has("email")))  # a let and implies

Person = S.OfObject.Builder().properties(text("name"), text("age", int), text("email")).create()
Proxies.register("Person", Person)
B = Proxies.Builders.Person
people = [B().name("Ann").age(30).email("a@x").create(), B().name("Bob").age(70).create(),
          B().name("x").age(10).email("c@x").create()]

## TRN-01 · Rules pair patterns; holes bind arguments and attributes

In [ ]:
A, B_, K = *holes("A", "B"), Hole("K", str)
double = P("operation", A, A, name="add")
assert double.size() == 2 and repr(double) == "operation(A, A, name='add')" and double.holes() == {"A": "argument"}
has = P("operation", A, P("literal", value=K), name="has")
assert has.size() == 3 and has.holes() == {"A": "argument", "K": "attribute"}
with raises(ValueError, match="operation(A, B, name='f') needs the argument hole B, which operation(A, name='f') does not bind"):
    Rule(P("operation", A, name="f"), P("operation", A, B_, name="f"))
with raises(ValueError, match="variable(name=A) needs the attribute hole A"):
    Rule(P("operation", A, name="f"), P("variable", name=A))
Rule(P("operation", A, name="f"), P("operation", A, B_, name="f"), "backward")  # only the other way needs B
with raises(ValueError, match="direction must be 'both', 'forward' or 'backward', got 'up'"):
    Rule(double, double, "up")
with raises(ValueError, match="side must be 'left' or 'right', got 'up'"):
    Inline("let", "up")
assert [(r.left.attributes, r.right.attributes) for r in renames("operation", "name", "call", "function",
                                                                  {"eq": "equal"}, 2)] == [
    ({"name": "eq"}, {"function": "equal"})]

# A translator within one dialect is a rewriting: holes bound twice must bind the same node, or equal values.
simplify = Pairwise(BASIC, BASIC, [
    Rule(double, P("operation", A, P("literal", value=2), name="mul"), "forward"),
    Rule(P("operation", P("literal", value=K), P("literal", value=K), name="eq"), P("literal", value=True), "forward"),
    Rule(P("operation", A, B_, name="add"), P("operation", A, B_, name="add"), "forward"),
    Rule(P("operation", P("literal", value=Hole("V")), P("literal", value=Hole("W")), name="eq"),
         P("literal", value=False), "forward"),
    Rule(P("variable", name=Hole("N")), P("variable", name=Hole("N"))),
    Rule(P("literal", value=Hole("V")), P("literal", value=Hole("V")), "forward"),
])
x, y = E.variable("x"), E.variable("y")
assert F.same(simplify.forward(x.add(x)), x.mul(2).data) and F.same(simplify.forward(x.add(y)), x.add(y).data)
assert simplify.forward(E.literal("a").eq("a")).value is True and simplify.forward(E.literal("a").eq("b")).value is False
assert simplify.forward(E.literal(1).eq(1)).value is False  # K holds only a str
assert repr(simplify) == "<translator Basic <-> Basic>"

## TRN-02 · Every pair translates both ways; forms round-trip where both dialects have them

In [ ]:
assert len(Translators.TRANSLATORS) == 6
for source, target in itertools.permutations(DIALECTS, 2):
    translator = Translators.between(source, target)
    assert translator.left() is source and translator.right() is target
    start = Translators.between(BASIC, source).forward(core) if source is not BASIC else core.data
    there = translator.forward(start)
    assert there.dialect() is target and there.validate(bound={"this"}, core=True) == [], (source.name(), target.name())
    assert F.same(translator.backward(there), start), (source.name(), target.name())  # round trip
    assert F.same(translator.inverse().backward(start), there)
with raises(LookupError, match="no translator between Basic and Basic"):
    Translators.between(BASIC, BASIC)

assert N.render(Translators.between(BASIC, NUMPY).forward(rule)) == (
    "np.where(np.greater_equal(this['age'], 18), np.logical_not(np.ma.getmaskarray(this['email'])), True)")
assert M.render(Translators.between(BASIC, MATLAB).forward(rule)) == '~(this.age >= 18) || isfield(this, "email")'
assert X.render(Translators.between(BASIC, EXCEL).forward(rule)) == (
    "=LET(a, this.age, IF(a >= 18, NOT(ISERROR(this.email)), TRUE))")
# Excel has lets and IF, so the rule round-trips; Numpy and Matlab inline the let, and Matlab writes implies with ||.
assert F.same(Translators.between(EXCEL, BASIC).forward(Translators.between(BASIC, EXCEL).forward(rule)), rule.data)
back = Translators.between(MATLAB, BASIC).forward(Translators.between(BASIC, MATLAB).forward(rule))
assert F.same(back, age.ge(18).not_().or_(this.has("email")).data)

## TRN-03 · Translating the same pair of dialects by any route gives the same expression

In [ ]:
for a, b, c in itertools.permutations(DIALECTS, 3):
    start = Translators.between(BASIC, a).forward(core) if a is not BASIC else core.data
    direct = Translators.between(a, c).forward(start)
    routed = Translators.between(b, c).forward(Translators.between(a, b).forward(start))
    assert F.same(direct, routed), (a.name(), b.name(), c.name())

## TRN-04 · Co-traversal translates each node once, keeps sharing, and traces the correspondence

In [ ]:
shared = E.operation("add", age, age).data
trace = []
out = Translators.between(BASIC, EXCEL).forward(shared, trace)
assert out.left is out.right  # the shared argument is translated once
assert [(type(s).KIND, type(t).KIND) for s, t in trace] == [("variable", "name"), ("operation", "field"),
                                                             ("operation", "infix")]
assert trace[-1] == (shared, out) and trace[1][1] is out.left
a = E.variable("a")
inlined = Translators.between(BASIC, NUMPY).forward(E.let_("a", age, a.add(a)), trace := [])
assert N.render(inlined) == "np.add(this['age'], this['age'])" and inlined.arguments[0] is inlined.arguments[1]
assert [type(s).KIND for s, _ in trace] == ["variable", "operation", "variable", "operation", "let"]
shadowed = E.let_("a", 1, E.variable("a").add(E.let_("a", 2, E.variable("a"))))  # the innermost binding wins
assert N.render(Translators.between(BASIC, NUMPY).forward(shadowed)) == "np.add(1, 2)"
assert NV.OfAny(Translators.between(BASIC, NUMPY).forward(shadowed)) == V.OfAny(shadowed) == 3
assert M.render(Translators.between(EXCEL, MATLAB).forward(X.let_("n", 2, X.infix("*", X.name("n"), X.name("n"))))) == "2 .* 2"
loop = E.OfOperation.Builder().name("not").create()
E.OfOperation.Builder(loop).arguments(loop).update()
with raises(ValueError, match="the expression contains a cycle"):
    Translators.between(BASIC, NUMPY).forward(loop)
with raises(TypeError, match="not an expression: 7"):
    Translators.between(BASIC, NUMPY).forward(E.OfOperation.Data("not", (7,)))
with raises(ValueError, match="a let needs a value"):
    Translators.between(BASIC, MATLAB).forward(E.OfLet.Data("a"))
with raises(ValueError, match="a let needs a body"):
    Translators.between(EXCEL, NUMPY).forward(X.DIALECT.kinds()["let"]("a", X.constant(1)))

## TRN-05 · What a dialect cannot say is refused, naming the node

In [ ]:
for translate, expression, message in [
    (Translators.between(BASIC, EXCEL).forward, E.literal(b"\x00"), "Basic literal b'\\x00' has no Excel counterpart"),
    (Translators.between(BASIC, NUMPY).forward, E.OfLiteral.Data(), "Basic literal None has no Numpy counterpart"),
    (Translators.between(BASIC, MATLAB).forward, E.operation("in", 1, 2), "Basic operation 'in' has no Matlab counterpart"),
    (Translators.between(BASIC, NUMPY).forward, this.get(E.variable("key")), "Basic operation 'get' has no Numpy counterpart"),
    (Translators.between(BASIC, NUMPY).forward, E.OfOperation.Data("has", (this.data, 7)), "Basic operation 'has'"),
    (Translators.between(BASIC, NUMPY).forward, E.operation("eq", 1, 2, 3), "Basic operation 'eq' has no Numpy counterpart"),
    (Translators.between(EXCEL, BASIC).forward, X.function("ISERROR", 1), "Excel function 'ISERROR' has no Basic counterpart"),
    (Translators.between(EXCEL, BASIC).forward, X.function("IF", True, 1, 2), "Excel function 'IF' has no Basic counterpart"),
    (Translators.between(EXCEL, BASIC).forward, X.DIALECT.kinds()["let"]("a", X.constant(1)), "Excel let has no Basic counterpart"),
    (Translators.between(NUMPY, MATLAB).forward, N.call("where", True, 1, 2), "Numpy call 'where' has no Matlab counterpart"),
    (Translators.between(MATLAB, BASIC).forward, M.call("isfield", M.identifier("s"), M.identifier("k")), "Matlab call 'isfield'"),
]:
    with raises(ValueError, match=message):
        translate(expression)

## TRN-06 · Translations evaluate alike where the dialects' rules agree, and apart where they do not

In [ ]:
translations = {d: Translators.between(BASIC, d).forward(core) if d is not BASIC else core.data for d in DIALECTS}
lets = {d: Translators.between(BASIC, d).forward(rule) if d is not BASIC else rule.data for d in DIALECTS}
for person in people:
    for expressions in (translations, lets):
        values = {d.name(): bool(EVALUATE[d](e, {"this": person})) for d, e in expressions.items()}
        assert len(set(values.values())) == 1, (person.name, values)
columns = {"age": np.array([30, 70, 10]), "name": np.array(["Ann", "Bob", "x"]),
           "email": np.ma.array(["a@x", "", "c@x"], mask=[False, True, False])}
assert NV.OfAny(translations[NUMPY], {"this": columns}).tolist() == [
    bool(V.OfAny(core, {"this": person})) for person in people]  # numpy evaluates all three at once

nobody = B().name("Nobody").create()  # no age: each dialect's own notion of a missing value
missing = age.ge(18)
assert V.OfAny(missing, {"this": nobody}) is None  # Basic: unknown
assert NV.OfAny(Translators.between(BASIC, NUMPY).forward(missing), {"this": nobody}) is np.ma.masked  # masked
assert XV.OfAny(Translators.between(BASIC, EXCEL).forward(missing), {"this": nobody}) == XV.Error("#FIELD!")  # an error value
with raises(KeyError, match='Unrecognized field name "age".'):  # MATLAB raises
    MV.OfAny(Translators.between(BASIC, MATLAB).forward(missing), {"this": nobody})
mixed, cased = E.literal(1).eq(1.0), E.literal("a").eq("A")
assert [V.OfAny(mixed), V.OfAny(cased)] == [None, False]  # Basic never coerces, and compares text exactly
assert [bool(EVALUATE[d](Translators.between(BASIC, d).forward(mixed))) for d in DIALECTS[1:]] == [True, True, True]
assert [bool(EVALUATE[d](Translators.between(BASIC, d).forward(cased))) for d in DIALECTS[1:]] == [False, False, True]